In [ ]:
import torch
import torch.nn.functional as F
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from definitions import ROOT_DIR, MPW_CNN_DIR
import src.cnn_utils

# Shallow SmallCNN

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = F.relu(self.conv3(x))      # last conv layer -> Grad-CAM target
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)                 # raw logits
        return x

# Transforms and loaders


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

img_size = 224
batch_size = 32

transformer = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5]),
])

full_dataset = datasets.ImageFolder("../data/icosimal_img_class_03/train", transform=transformer)
splits = torch.load("../data/split/split_train_test_indices.pth")

train_ds = torch.utils.data.Subset(full_dataset, splits['train_idx'])
test_ds = torch.utils.data.Subset(full_dataset, splits['test_idx'])
val_ds = datasets.ImageFolder("../data/icosimal_img_class_03/validate", transform=transformer)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)

print(f"Training dataset size: {len(train_ds)}")
print(f"Validation dataset size: {len(val_ds)}")
print(f"Test dataset size: {len(test_ds)}")

class_names = val_ds.classes
num_classes = len(class_names)
class_to_idx = val_ds.class_to_idx
print("Number of classes: ", num_classes)
print("Class names: ", class_names)

# Device

In [ ]:
# Check for GPU
device = None
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Current device : ", device)

# Model, loss, optimizer

In [ ]:
model = SmallCNN(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Train

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

# Loop

In [ ]:
num_epochs = 10
best_val_acc = 0.0

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    print(
        f"Epoch {epoch+1:02d}/{num_epochs} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_smallcnn.pt")

print("Best val acc:", best_val_acc)

# Grad-CAM evaluation and visualization
## Load best model

In [ ]:
model = SmallCNN(num_classes=num_classes).to(device)
model.load_state_dict(torch.load("best_smallcnn.pt", map_location=device))
model.eval()



In [ ]:
batch_size = 32
nepochs = 10
lr = 0.1
units = 100



optimizer = torch.optim.SGD(params=model.parameters(), lr = lr)
cost_train_sgd, cost_valid_sgd, acc_train_sgd, acc_valid_sgd = (
    src.cnn_utils.train_eval(model, optimizer, nepochs, batch_size, train_ds, val_ds, device, entity='MSE_DeLearn_SPR26', project='MPW-CNN', run_name='small_CNN', use_wandb=True))

In [ ]:
# import cv2
# import numpy as np
# from pytorch_grad_cam.utils.image import preprocess_image
#
# file_name = "0d3211c08b0095d66b7dedaaaa451ab5.jpg"
#
# rgb_img = cv2.imread("../data/icosimal_img_class_03/validate/cat/" + file_name)[:, :, ::-1]
# rgb_img = cv2.resize(rgb_img, (img_size, img_size))
# rgb_img = np.float32(rgb_img) / 255.0
#
# input_tensor = preprocess_image(
#     rgb_img,
#     mean=[0.5, 0.5, 0.5],
#     std=[0.5, 0.5, 0.5]
# ).to(device)

## Inference


In [ ]:
# with torch.no_grad():
#     logits = model(input_tensor)
#     pred_class = int(logits.argmax(dim=1).item())
#     confidence = float(torch.softmax(logits, dim=1)[0, pred_class].item())
#
# print("Predicted: ", class_names[pred_class], " with confidence: ", confidence)

## Grad-CAM

In [ ]:
import os
from pathlib import Path

import cv2
import numpy as np
import torch

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import preprocess_image, show_cam_on_image


def inspect_image_with_gradcam_from_folder_label(
    model,
    image_path,
    target_layers,
    class_names,
    class_to_idx,
    mean,
    std,
    device=None,
    resize_to=None,
    output_dir="gradcam_outputs",
    eigen_smooth=False,
    aug_smooth=False,
):
    """
    Assumes image_path looks like:
        .../<split>/<class_name>/<image_file>

    True class is inferred from the parent folder name.
    """

    if device is None:
        device = next(model.parameters()).device

    model.eval()
    image_path = Path(image_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # ---- infer true class from parent folder ----
    true_class_name = image_path.parent.name
    if true_class_name not in class_to_idx:
        raise ValueError(
            f"Parent folder '{true_class_name}' is not in class_to_idx: {list(class_to_idx.keys())}"
        )
    true_class = class_to_idx[true_class_name]

    # ---- read image ----
    bgr = cv2.imread(str(image_path))
    if bgr is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")

    rgb_img = bgr[:, :, ::-1]
    if resize_to is not None:
        rgb_img = cv2.resize(rgb_img, resize_to)

    rgb_img = np.float32(rgb_img) / 255.0

    # ---- preprocess ----
    input_tensor = preprocess_image(
        rgb_img,
        mean=mean,
        std=std
    ).to(device)

    # ---- predict ----
    with torch.no_grad():
        logits = model(input_tensor)
        probs = torch.softmax(logits, dim=1)

        pred_class = int(logits.argmax(dim=1).item())
        pred_conf = float(probs[0, pred_class].item())
        true_conf = float(probs[0, true_class].item())
        is_correct = (pred_class == true_class)

    result = {
        "image_path": str(image_path),
        "true_class_idx": true_class,
        "true_class_name": class_names[true_class],
        "true_class_confidence": true_conf,
        "pred_class_idx": pred_class,
        "pred_class_name": class_names[pred_class],
        "pred_confidence": pred_conf,
        "is_correct": is_correct,
        "saved_files": [],
    }

    stem = image_path.stem

    with GradCAM(model=model, target_layers=target_layers) as cam:
        if is_correct:
            # one heatmap only
            grayscale_cam = cam(
                input_tensor=input_tensor,
                targets=[ClassifierOutputTarget(pred_class)],
                eigen_smooth=eigen_smooth,
                aug_smooth=aug_smooth,
            )[0, :]

            overlay = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
            out_path = output_dir / f"{stem}__correct_{class_names[pred_class]}.jpg"
            cv2.imwrite(str(out_path), overlay[:, :, ::-1])
            result["saved_files"].append(str(out_path))

        else:
            # predicted-class heatmap
            cam_pred = cam(
                input_tensor=input_tensor,
                targets=[ClassifierOutputTarget(pred_class)],
                eigen_smooth=eigen_smooth,
                aug_smooth=aug_smooth,
            )[0, :]

            overlay_pred = show_cam_on_image(rgb_img, cam_pred, use_rgb=True)
            out_pred = output_dir / f"{stem}__pred_{class_names[pred_class]}.jpg"
            cv2.imwrite(str(out_pred), overlay_pred[:, :, ::-1])
            result["saved_files"].append(str(out_pred))

            # true-class heatmap
            cam_true = cam(
                input_tensor=input_tensor,
                targets=[ClassifierOutputTarget(true_class)],
                eigen_smooth=eigen_smooth,
                aug_smooth=aug_smooth,
            )[0, :]

            overlay_true = show_cam_on_image(rgb_img, cam_true, use_rgb=True)
            out_true = output_dir / f"{stem}__true_{class_names[true_class]}.jpg"
            cv2.imwrite(str(out_true), overlay_true[:, :, ::-1])
            result["saved_files"].append(str(out_true))

    return result

In [ ]:
img_path = os.path.join(MPW_CNN_DIR, "data", "icosimal_img_class_03", "validate", "cat", "1a63f354a05e5ce823d7b0308394c451.jpg")

result = inspect_image_with_gradcam_from_folder_label(
    model=model,
    image_path=img_path,
    target_layers=[model.conv3],   # replace if your last conv layer is different
    class_names=val_ds.classes,
    class_to_idx=val_ds.class_to_idx,
    mean=[0.5, 0.5, 0.5],
    std=[0.5, 0.5, 0.5],
    device=device,
    resize_to=(128, 128),
    output_dir="gradcam_outputs",
)

print("predicted :", result["pred_class_name"], result["pred_confidence"])
print("true      :", result["true_class_name"], result["true_class_confidence"])
print("correct   :", result["is_correct"])
print("files     :", result["saved_files"])

## Model metrics and confusion matrix

In [ ]:

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

# expects:
# - model already loaded and in eval() mode
# - val_loader
# - class_names (e.g. train_ds.classes)
# - device

@torch.no_grad()
def evaluate_on_loader(model, loader, class_names, device):
    model.eval()

    all_true = []
    all_pred = []
    all_probs = []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        all_true.extend(labels.cpu().numpy())
        all_pred.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    y_true = np.array(all_true)
    y_pred = np.array(all_pred)
    y_prob = np.array(all_probs)

    # overall accuracy
    acc = accuracy_score(y_true, y_pred)
    print(f"Validation accuracy: {acc:.4f}")

    # text metrics
    print("\nClassification report:")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

    # confusion matrices
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = confusion_matrix(y_true, y_pred, normalize="true")

    # per-class accuracy = recall per class = diag / row sum
    per_class_acc = np.divide(
        np.diag(cm),
        cm.sum(axis=1),
        out=np.zeros(len(class_names), dtype=float),
        where=cm.sum(axis=1) != 0
    )

    # confidence of predicted class
    pred_conf = y_prob[np.arange(len(y_pred)), y_pred]
    correct_mask = (y_true == y_pred)

    # ---- plots ----

    # 1) per-class accuracy bar chart
    plt.figure(figsize=(10, 4))
    plt.bar(class_names, per_class_acc)
    plt.ylim(0, 1)
    plt.ylabel("Per-class accuracy")
    plt.title(f"Validation accuracy = {acc:.4f}")
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis="y")
    plt.tight_layout()
    plt.show()
    plt.tight_layout()
    plt.show()

    # 2) raw confusion matrix
    fig, ax = plt.subplots(figsize=(7, 7))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, xticks_rotation=45, colorbar=False, cmap="Blues")
    ax.set_title("Confusion Matrix (Counts)")
    plt.tight_layout()
    plt.show()

    # 3) normalized confusion matrix
    fig, ax = plt.subplots(figsize=(7, 7))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=class_names)
    disp.plot(ax=ax, xticks_rotation=45, values_format=".2f", colorbar=False, cmap="Blues")
    ax.set_title("Confusion Matrix (Row-normalized)")
    plt.tight_layout()
    plt.show()

    # 4) confidence histogram: correct vs wrong
    plt.figure(figsize=(8, 4))
    plt.hist(pred_conf[correct_mask], bins=20, alpha=0.7, label="Correct")
    plt.hist(pred_conf[~correct_mask], bins=20, alpha=0.7, label="Wrong")
    plt.xlabel("Predicted-class confidence")
    plt.ylabel("Count")
    plt.title("Confidence Distribution")
    plt.legend()
    plt.tight_layout()
    plt.grid(axis="y")
    plt.show()

    # top-N most confident mistakes
    N = 10
    pred_conf = y_prob[np.arange(len(y_pred)), y_pred]
    wrong_idx = np.where(y_true != y_pred)[0]
    top_wrong = wrong_idx[np.argsort(pred_conf[wrong_idx])[::-1][:N]]

    print("Most confident mistakes:")
    for i in top_wrong:
        print(
            f"idx={i:4d}  true={class_names[y_true[i]]:>15s}  "
            f"pred={class_names[y_pred[i]]:>15s}  conf={pred_conf[i]:.4f}"
        )

    return {
        "accuracy": acc,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
        "cm": cm,
        "cm_norm": cm_norm,
        "per_class_acc": per_class_acc,
    }

# run it
results = evaluate_on_loader(model, val_loader, class_names, device)